### Step 1: Install necesscary packages

### Step 2: Package imports and configuration

In [21]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 15
batch_size = 64
max_length = 128
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [22]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [23]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 5: Load Data (**students are required to complete this part!**)

In [24]:
# Load data from ./pos_neg_pairs_test.json (or ./pos_neg_pairs.json)
data_path = "./pos_neg_pairs_test.json"
with open(data_path, "r", encoding="utf-8") as f:
    lines = json.load(f)


In [14]:
len(lines)

92953

### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [25]:
# recommend to use the AdamW optimizer
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr, weight_decay=0.01)
total_training_steps = max(1, (len(lines) // batch_size) * epochs)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_training_steps, eta_min=base_lr * 0.1)


### Step 7: Begin training (**students are required to complete this part!**)

In [26]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size), total=total_steps)
    for step, (neg_tensor, pos_tensor) in enumerate(pbar):
        optimizer.zero_grad()
        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        pbar.set_description(f"Epoch {epoch + 1}/{epochs} | Loss: {loss.item():.4f}")
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

Epoch 1/15 | Loss: 0.0282: 100%|██████████| 1452/1452 [01:01<00:00, 23.63it/s]


Saved checkpoint to ./dpo.pt


Epoch 2/15 | Loss: 0.0261: 100%|██████████| 1452/1452 [01:01<00:00, 23.61it/s]


Saved checkpoint to ./dpo.pt


Epoch 3/15 | Loss: 0.0228: 100%|██████████| 1452/1452 [01:01<00:00, 23.59it/s]


...

Saved checkpoint to ./dpo.pt


Epoch 14/15 | Loss: 0.0186: 100%|██████████| 1452/1452 [01:01<00:00, 23.61it/s]


Saved checkpoint to ./dpo.pt


Epoch 15/15 | Loss: 0.0184: 100%|██████████| 1452/1452 [01:01<00:00, 23.60it/s]

Saved checkpoint to ./dpo.pt


### Step 8: Begin testing (**students are required to complete this part!**)

In [27]:
# string match
def extract_answer(s):
    # find "so the answer is {ans}." and return ans
    return s.split("so the answer is ")[-1].split(".")[0].strip()

from data_generator import generate_random_problems, Problem

test_set_problems: list[Problem] = generate_random_problems(500)

In [28]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()

correct = 0
wrong = 0

with torch.no_grad():
    for problem in test_set_problems:
        prompt = problem.question_string
        prompt_ids = encode(prompt)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]
        y = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        resp = decode(y[0].squeeze().tolist())
        if extract_answer(resp) == str(problem.solution):
            correct += 1
        else:
            print(resp)
            wrong += 1

print(f"Correct: {correct}, Wrong: {wrong}, Accuracy: {correct / (correct + wrong):.4f}")


23*52=? 23*52 equals 1286, so the answer is 1286.
95*x=4465,x=? x=4465/95, and 4465/95 equals 49, so the answer is 49.
19*37=? 19*37 equals 753, so the answer is 753.
91*34=? 91*34 equals 3144, so the answer is 3144.
35*48=? 35*48 equals 1570, so the answer is 1570.
81*46=? 81*46 equals 3506, so the answer is 3506.
14*26=? 14*26 equals 344, so the answer is 344.
53*2=? 53*2 equals 116, so the answer is 116.
67*87=? 67*87 equals 5609, so the answer is 5609.
x/2=35,x=? x=35*2, and 35*2 equals 60, so the answer is 60.
35*73=? 35*73 equals 2525, so the answer is 2525.
44*71=? 44*71 equals 3114, so the answer is 3114.
79*6=? 79*6 equals 484, so the answer is 484.
57*94=? 57*94 equals 5588, so the answer is 5588.
58*60=? 58*60 equals 3680, so the answer is 3680.
13*21=? 13*21 equals 293, so the answer is 293.
50*46=? 50*46 equals 2400, so the answer is 2400.
18-21=? 18-21 equals -5, so the answer is -5.
70*17=? 70*17 equals 1290, so the answer is 1290.
73*82=? 73*82 equals 6156, so the answe